## Reading in a short story as text sample into Python.

## Step 1: Creating Tokens

The print command prints the total number of characters followed by the first 100
characters of this file for illustration purposes

In [95]:
with open("data/verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20240
I had always thought Jack Gisburn rather a cheap genius—though a good fellow enough—so it was no gr


Our goal is to tokenize this 20,240-character short story into individual words and special
characters that we can then turn into embeddings for LLM training 

Note that it's common to process millions of articles and hundreds of thousands of
books -- many gigabytes of text -- when working with LLMs. However, for educational
purposes, it's sufficient to work with smaller text samples like a single book to
illustrate the main ideas behind the text processing steps and to make it possible to
run it in reasonable time on consumer hardware.

How can we best split this text to obtain a list of tokens? For this, we go on a small
excursion and use Python's regular expression library re for illustration purposes. (Note
that you don't have to learn or memorize any regular expression syntax since we will
transition to a pre-built tokenizer later in this chapter.)

In [96]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


Let's modify the regular expression splits on whitespaces (\s) and commas, and periods
([,.]):

In [97]:
result = re.split(r'([.,]|\s)', text)

print(result)


['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


Remove the whitespaces

In [98]:
result = [token for token in result if token.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


REMOVING WHITESPACES OR NOT


When developing a simple tokenizer, whether we should encode whitespaces as
separate characters or just remove them depends on our application and its
requirements. Removing whitespaces reduces the memory and computing
requirements. However, keeping whitespaces can be useful if we train models that
are sensitive to the exact structure of the text (for example, Python code, which is
sensitive to indentation and spacing). Here, we remove whitespaces for simplicity
and brevity of the tokenized outputs. Later, we will switch to a tokenization scheme
that includes whitespaces.

The tokenization scheme I devised above works well on the simple sample text. Let's
modify it a bit further so that it can also handle other types of punctuation, such as
question marks, quotation marks, and the double-dashes we have seen earlier in the first
100 characters of Edith Wharton's short story, along with additional special characters:

In [99]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


Now that we got a basic tokenizer working, let's apply it to Edith Wharton's entire short
story:

In [100]:
preprocessed = re.split(r'([,.:;?_!"—()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'had', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '—', 'though', 'a', 'good', 'fellow', 'enough', '—', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [101]:
print("Total number of tokens:", len(preprocessed))

Total number of tokens: 4396


## Step 2: Creating Token IDs

In the previous section, we tokenized Edith Wharton's short story and assigned it to a Python variable called preprocessed. Let's now create a list of all unique tokens and sort them alphabetically to determine the vocabulary size:

In [102]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print("Total number of unique words:", vocab_size)

Total number of unique words: 1181


After determining that the vocabulary size is 1,181 via the above code, we create the
vocabulary and print its first 51 entries for illustration purposes:

In [103]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [104]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('(', 1)
(')', 2)
(',', 3)
('.', 4)
(':', 5)
(';', 6)
('?', 7)
('A', 8)
('Among', 9)
('And', 10)
('Arrt', 11)
('As', 12)
('At', 13)
('Burlington', 14)
('But', 15)
('By', 16)
('Carlo', 17)
('Chicago', 18)
('Claude', 19)
('Croft', 20)
('Devonshire', 21)
('Don’t', 22)
('Dubarry', 23)
('Emperors', 24)
('Florence', 25)
('For', 26)
('Gallery', 27)
('Gideon', 28)
('Gisburn', 29)
('Gisburn’s', 30)
('Grafton', 31)
('Greek', 32)
('Grindle', 33)
('Grindle’s', 34)
('Had', 35)
('He', 36)
('Her', 37)
('Hermia', 38)
('Hermia’s', 39)
('His', 40)
('I', 41)
('If', 42)
('In', 43)
('It', 44)
('I’d', 45)
('I’ll', 46)
('I’ve', 47)
('Jack', 48)
('Jack’s', 49)
('Jove', 50)


As we can see, based on the output above, the dictionary contains individual tokens associated with unique integer labels.

Later in this book, when we want to convert the outputs of an LLM from numbers back into text, we also need a way to turn token IDs into text.

For this, we can create an inverse version of the vocabulary that maps token IDs back to corresponding text tokens.

Let's implement a complete tokenizer class in Python.

The class will have an encode method that splits text into tokens and carries out the string-to-integer mapping to produce token IDs via the vocabulary.

In addition, we implement a decode method that carries out the reverse integer-to-string mapping to convert the token IDs back into text.

Step 1: Store the vocabulary as a class attribute for access in the encode and decode methods

Step 2: Create an inverse vocabulary that maps token IDs back to the original text tokens

Step 3: Process input text into token IDs

Step 4: Convert token IDs back into text

Step 5: Replace spaces before the specified punctuation

In [105]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([,.:;?_!"—()\']|--|\s)', text)
        tokens = [token.strip() for token in tokens if token.strip()]
        ids = [self.str_to_int[token] for token in tokens]
        return ids

    def decode(self, token_ids):
        text = ' '.join([self.int_to_str[token_id] for token_id in token_ids])
        # replace spaces before punctuation
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

Let's instantiate a new tokenizer object from the SimpleTokenizerV1 class and tokenize a passage from Edith Wharton's short story to try it out in practice

In [106]:
tokenizer = SimpleTokenizerV1(vocab)

text = """I turned to Mrs. Gisburn, who had lingered to give a lump of sugar to her spaniel in the dining-room.
“Why has he chucked painting?” I asked abruptly."""
ids = tokenizer.encode(text)
print(ids)

[41, 1023, 1002, 57, 4, 29, 3, 1084, 500, 624, 1002, 475, 102, 641, 712, 939, 1002, 525, 903, 558, 972, 327, 4, 1167, 515, 519, 249, 738, 7, 1180, 41, 167, 107, 4]



The code above prints the following token IDs: Next, let's see if we can turn these token IDs back into text using the decode method:

In [107]:
tokenizer.decode(ids)

'I turned to Mrs. Gisburn, who had lingered to give a lump of sugar to her spaniel in the dining-room. “Why has he chucked painting? ” I asked abruptly.'

Based on the output above, we can see that the decode method successfully converted the token IDs back into the original text.

So far, so good. Implemented a tokenizer capable of tokenizing and de-tokenizing text based on a snippet from the training set.

Let's now apply it to a new text sample that is not contained in the training set:

In [108]:
text = 'Hello, Do you like to drink tea?'
try:
    print(tokenizer.encode(text))
except KeyError as e:
    print(f"Error: The token '{e.args[0]}' is not in the vocabulary.")

Error: The token 'Hello' is not in the vocabulary.


The problem is that the word "Hello" was not used in the The Verdict short story.

Hence, it is not contained in the vocabulary.

This highlights the need to consider large and diverse training sets to extend the vocabulary when working on LLMs.

ADDING SPECIAL CONTEXT TOKENS
In the previous section, we implemented a simple tokenizer and applied it to a passage from the training set.

In this section, we will modify this tokenizer to handle unknown words.

In particular, we will modify the vocabulary and tokenizer we implemented in the previous section, SimpleTokenizerV2, to support two new tokens, <|unk|> and <|endoftext|>

We can modify the tokenizer to use an <|unk|> token if it encounters a word that is not part of the vocabulary.

Furthermore, we add a token between unrelated texts.

For example, when training GPT-like LLMs on multiple independent documents or books, it is common to insert a token before each document or book that follows a previous text source

Let's now modify the vocabulary to include these two special tokens, and <|endoftext|>, by adding these to the list of all unique words that we created in the previous section:

In [109]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [110]:
print('Vocabulary size after adding special tokens:', len(vocab.items()))

Vocabulary size after adding special tokens: 1183


Based on the output of the print statement above, the new vocabulary size is 1183 (the vocabulary size in the previous section was 1181).

As an additional quick check, let's print the last 5 entries of the updated vocabulary:

In [111]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('“strongly”', 1178)
('“sweetly”', 1179)
('”', 1180)
('<|endoftext|>', 1181)
('<|unk|>', 1182)


A simple text tokenizer that handles unknown words

Step 1: Replace unknown words by <|unk|> tokens

Step 2: Replace spaces before the specified punctuations

In [112]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([,.:;?_!"—()\']|--|\s)', text)
        tokens = [token.strip() for token in tokens if token.strip()]
        tokens = [token if token in self.str_to_int else "<|unk|>" for token in tokens]
        ids = [self.str_to_int[token] for token in tokens]
        return ids

    def decode(self, token_ids):
        text = ' '.join([self.int_to_str[token_id] for token_id in token_ids])
        # replace spaces before punctuation
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [113]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [114]:
tokenizer.encode(text)

[1182, 3, 341, 1114, 620, 960, 7, 1181, 43, 972, 943, 968, 712, 972, 1182, 4]

In [115]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

Based on comparing the de-tokenized text above with the original input text, we know that the training dataset, Edith Wharton's short story The Verdict, did not contain the words "Hello" and "palace."

So far, we have discussed tokenization as an essential step in processing text as input to LLMs. Depending on the LLM, some researchers also consider additional special tokens such as the following:

[BOS] (beginning of sequence): This token marks the start of a text. It signifies to the LLM where a piece of content begins.

[EOS] (end of sequence): This token is positioned at the end of a text, and is especially useful when concatenating multiple unrelated texts, similar to <|endoftext|>. For instance, when combining two different Wikipedia articles or books, the [EOS] token indicates where one article ends and the next one begins.

[PAD] (padding): When training LLMs with batch sizes larger than one, the batch might contain texts of varying lengths. To ensure all texts have the same length, the shorter texts are extended or "padded" using the [PAD] token, up to the length of the longest text in the batch.

Note that the tokenizer used for GPT models does not need any of these tokens mentioned above but only uses an <|endoftext|> token for simplicity

the tokenizer used for GPT models also doesn't use an <|unk|> token for outof-vocabulary words. Instead, GPT models use a byte pair encoding tokenizer, which breaks down words into subword units

Trying this tokenizer on another books

In [116]:
with open("data/alice_in_wonderland.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

raw_text[:100]

'Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to'

In [117]:
len(raw_text)

32934

In [118]:
words = re.split(r'([,.:;?_!"—()\']|--|\s)', raw_text)
words = [item.strip() for item in words if item.strip()]

print('Total number of words:', len(words))
print('Sample words:', words[:10])

Total number of words: 7564
Sample words: ['Alice', 'was', 'beginning', 'to', 'get', 'very', 'tired', 'of', 'sitting', 'by']


In [119]:
unique_words = sorted(set(words))
print('Total number of unique words:', len(unique_words))
vocab = {token:integer for integer,token in enumerate(unique_words)}

Total number of unique words: 1043


In [120]:
print('Vocabulary size:', len(vocab))
for i, item in enumerate(list(vocab.items())):
    print(item)
    if i >= 50:
        break

Vocabulary size: 1043
('!', 0)
('(', 1)
(')', 2)
(',', 3)
('.', 4)
(':', 5)
(';', 6)
('?', 7)
('A', 8)
('Ada', 9)
('After', 10)
('Ah', 11)
('Alice', 12)
('Alice’s', 13)
('And', 14)
('Antipathies', 15)
('As', 16)
('Australia', 17)
('Besides', 18)
('But', 19)
('Christmas', 20)
('Conqueror', 21)
('Dinah', 22)
('Do', 23)
('Dodo', 24)
('Don’t', 25)
('Down', 26)
('Duchess', 27)
('Duck', 28)
('Eaglet', 29)
('Either', 30)
('English', 31)
('Esq', 32)
('Everything', 33)
('Fender', 34)
('First', 35)
('Foot', 36)
('For', 37)
('French', 38)
('Geography', 39)
('Good-bye', 40)
('Grammar', 41)
('He', 42)
('Hearthrug', 43)
('Her', 44)
('How', 45)
('However', 46)
('I', 47)
('Improve', 48)
('In', 49)
('It', 50)


In [121]:
tokenizer = SimpleTokenizerV1(vocab)

text = """I’m sure I’m not Ada,” se said, “for her hair goes in such long ringlets, and mine doesn’t go in ringlets at all; and I’m sure I can’t be Mabel, for I know all sorts of things, and she, oh! she knows such a very little! Besides, she’s she, and I’m I, and—oh dear, how puzzling it all is! I’ll try if I know all the things I used to know. Let me see: four times five is twelve, and four times six is thirteen, and four times seven is—oh dear! I shall never get to twenty at that rate! However, the Multiplication Table doesn’t signify: let’s try Geography. London is the capital of Paris, and Paris is the capital of Rome, and Rome—no, that’s all wrong, I’m certain! I must have been changed for Mabel! I’ll try and say ‘How doth the little—’” and she crossed her hands on her lap as if she were saying lessons, and began to repeat it, but her voice sounded hoarse and strange, and the words did not come the same as they used to do:—
“How doth the little crocodile"""

tokenizer.encode(text)

[53,
 827,
 53,
 598,
 9,
 3,
 1042,
 727,
 713,
 3,
 1033,
 436,
 411,
 398,
 469,
 823,
 533,
 706,
 3,
 129,
 564,
 285,
 397,
 469,
 706,
 143,
 119,
 6,
 129,
 53,
 827,
 47,
 199,
 153,
 65,
 3,
 370,
 47,
 493,
 119,
 798,
 608,
 858,
 3,
 129,
 746,
 3,
 611,
 0,
 746,
 495,
 823,
 103,
 917,
 527,
 0,
 18,
 3,
 750,
 746,
 3,
 129,
 53,
 47,
 3,
 129,
 985,
 611,
 266,
 3,
 456,
 676,
 475,
 119,
 474,
 0,
 52,
 893,
 466,
 47,
 493,
 119,
 848,
 858,
 47,
 912,
 877,
 493,
 4,
 58,
 556,
 731,
 5,
 376,
 873,
 362,
 474,
 899,
 3,
 129,
 376,
 873,
 773,
 474,
 861,
 3,
 129,
 376,
 873,
 740,
 474,
 985,
 611,
 266,
 0,
 47,
 743,
 588,
 390,
 877,
 900,
 143,
 846,
 688,
 0,
 46,
 3,
 848,
 68,
 89,
 285,
 766,
 5,
 519,
 893,
 39,
 4,
 59,
 474,
 848,
 200,
 608,
 76,
 3,
 129,
 76,
 474,
 848,
 200,
 608,
 83,
 3,
 129,
 83,
 985,
 595,
 3,
 847,
 119,
 975,
 3,
 53,
 210,
 0,
 47,
 574,
 426,
 158,
 213,
 370,
 65,
 0,
 52,
 893,
 129,
 721,
 987,
 293,
 848,
 527,
 985,

In [122]:
tokenizer.decode(tokenizer.encode(text))

'I’m sure I’m not Ada, ” se said, “for her hair goes in such long ringlets, and mine doesn’t go in ringlets at all ; and I’m sure I can’t be Mabel, for I know all sorts of things, and she, oh! she knows such a very little! Besides, she’s she, and I’m I, and — oh dear, how puzzling it all is! I’ll try if I know all the things I used to know. Let me see : four times five is twelve, and four times six is thirteen, and four times seven is — oh dear! I shall never get to twenty at that rate! However, the Multiplication Table doesn’t signify : let’s try Geography. London is the capital of Paris, and Paris is the capital of Rome, and Rome — no, that’s all wrong, I’m certain! I must have been changed for Mabel! I’ll try and say ‘How doth the little — ’” and she crossed her hands on her lap as if she were saying lessons, and began to repeat it, but her voice sounded hoarse and strange, and the words did not come the same as they used to do : — “How doth the little crocodile'

Based on comparing the de-tokenized text above with the original input text, we know that the training dataset, Edith Wharton's short story The Verdict, did not contain the words "Hello" and "palace."

So far, we have discussed tokenization as an essential step in processing text as input to LLMs. Depending on the LLM, some researchers also consider additional special tokens such as the following:

[BOS] (beginning of sequence): This token marks the start of a text. It signifies to the LLM where a piece of content begins.

[EOS] (end of sequence): This token is positioned at the end of a text, and is especially useful when concatenating multiple unrelated texts, similar to <|endoftext|>. For instance, when combining two different Wikipedia articles or books, the [EOS] token indicates where one article ends and the next one begins.

[PAD] (padding): When training LLMs with batch sizes larger than one, the batch might contain texts of varying lengths. To ensure all texts have the same length, the shorter texts are extended or "padded" using the [PAD] token, up to the length of the longest text in the batch.

Note that the tokenizer used for GPT models does not need any of these tokens mentioned above but only uses an <|endoftext|> token for simplicity

the tokenizer used for GPT models also doesn't use an <|unk|> token for outof-vocabulary words. Instead, GPT models use a byte pair encoding tokenizer, which breaks down words into subword units